### 🏛️ Notebook: Load Gold Layer (Star Schema & Business Intelligence)
---
#### 🎯 Purpose:
* **Dimensional Modeling:** Transform Silver data into a **Star Schema** (Facts & Dimensions) for optimized Power BI performance.
* **Surrogate Keys:** Generate unique sequential keys (`skey`) for dimensions to handle relationships efficiently.
* **Customer Segmentation:** Apply advanced logic to classify customers into segments (VIP, Loyal, New, At Risk, Lost).
* **Temporal Intelligence:** Enrich the Date dimension with **Day Parts** (Breakfast, Lunch, Dinner, etc.) based on order hours.
* **Geospatial & Visual Integration:** Join with Geo-data for branches and Photo-URLs for products.

---
#### 📂 Data Mapping (Star Schema):

| Target Table (Gold) | Type | Business Logic / Transformation |
| :--- | :--- | :--- |
| `dim_products` | **Dimension** | Aggregates prices and joins with Photo URLs. |
| `dim_branches` | **Dimension** | Joins with Geo-spatial data (WKT Geometry & Provinces). |
| `dim_date` | **Dimension** | Explodes dates into hierarchies + **Day Part** classification. |
| `dim_customers` | **Dimension** | Calculates **Recency (Days Gap)** and **Frequency** for Segmentation. |
| `dim_payment` | **Dimension** | Distinct payment methods with surrogate keys. |
| `dim_order_type` | **Dimension** | Distinct order channels (Delivery, Dine-in, etc.). |
| `fact_orders` | **Fact** | Central table linking all keys with metrics (Quantity, Price, Total). |

---
#### 🚀 Key Features:
* **ACID Compliance:** Data is saved in **Delta format** with `overwrite` mode for full refresh.
* **Performance:** Reordering columns to ensure Keys are at the beginning for faster indexing.
* **Observability:** Built-in logging for **Row Counts** and **Load Duration** for each table.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import time

In [0]:
# 1. Source and Target Tables
source_silver_table = "restaurant_project.silver.rest_sales"
target_dim_products = "restaurant_project.gold.dim_products"
target_dim_date     = "restaurant_project.gold.dim_date"
target_dim_branches = "restaurant_project.gold.dim_branches"
target_dim_payment  = "restaurant_project.gold.dim_payment"
target_dim_order_type = "restaurant_project.gold.dim_order_type"
source_silver_geo   = "restaurant_project.silver.branches_geo"
target_fact_orders  = "restaurant_project.gold.fact_orders"
source_silver_photo = "restaurant_project.silver.photo_url"


def load_gold_layer():
    try:
        batch_start_time = time.time()
        print('================================================')
        print('Loading Gold Layer (Star Schema)')
        print('================================================')

        # read data from silver layer
        df_silver = spark.read.table(source_silver_table)
        df_geo    = spark.read.table(source_silver_geo)
        df_photo = spark.read.table(source_silver_photo)

        # --- 1. Create Dimension: gold.dim_products ---
        print('------------------------------------------------')
        print('Loading dim_products Table')
        print('------------------------------------------------')
        start_time = time.time()

        dim_products_df = df_silver.groupBy("item_name", "category")\
            .agg(F.round(F.avg("price"), 2).alias("avg_unit_price"))

        # Sequential Key and Column Ordering
        prod_window = Window.orderBy("item_name")
        dim_products_df = dim_products_df.withColumn(
            "product_skey", F.row_number().over(prod_window)
        )
        
        dim_products_df = dim_products_df.join(
            df_photo,
            dim_products_df.product_skey == df_photo.photo_id,
            "left"
        ).drop(df_photo.photo_id)

        # Reorder: Key first, then descriptive columns
        dim_products_df = dim_products_df.select(
            "product_skey", "item_name", "category", "avg_unit_price", "photo_url"
        )

        # write dim_products to gold layer
        dim_products_df.write.format("delta").mode("overwrite").saveAsTable(target_dim_products)

        print(f'>> Rows Loaded: {dim_products_df.count()}')
        print(f'>> Load Duration: {round(time.time() - start_time, 2)} seconds')

        # --- 2. Create Dimension: gold.dim_branches ---
        print('------------------------------------------------')
        print('Loading dim_branches Table')
        print('------------------------------------------------')
        start_time = time.time()
        dim_branches_df = df_silver.select("branch").distinct().orderBy("branch")   

        # Sequential Key and Column Ordering
        branch_window = Window.orderBy("branch")
        dim_branches_df = dim_branches_df.withColumn(
            "branch_skey", F.row_number().over(branch_window)
        )

        dim_branches_final = dim_branches_df.join(
            df_geo, 
            dim_branches_df.branch_skey == df_geo.branch_id, 
            "left"
        ).drop(df_geo.branch_id)

        # Reorder: Key first, then branch name
        dim_branches_final = dim_branches_final.select(
            "branch_skey", "branch",  "province_en", "wkt_geometry")

        # write dim_branches to gold layer
        dim_branches_final.write.format("delta").mode("overwrite").saveAsTable(target_dim_branches)

        print(f'>> Rows Loaded: {dim_branches_final.count()}')
        print(f'>> Load Duration: {round(time.time() - start_time, 2)} seconds')

        # --- 3. Create Dimension: gold.dim_date ---
        print('------------------------------------------------')
        print('Loading dim_date Table')
        print('------------------------------------------------')
        start_time = time.time()

        dim_date_df = df_silver.select("order_date", "hour", "is_weekend").distinct()

        # Extract date parts and Reorder
        dim_date_df = dim_date_df \
            .withColumn("year", F.year("order_date"))\
            .withColumn("month", F.month("order_date"))\
            .withColumn("month_name", F.date_format("order_date", "MMMM"))\
            .withColumn("day", F.dayofmonth("order_date"))\
            .withColumn("day_of_week_number", F.dayofweek("order_date"))\
            .withColumn("day_name", F.date_format("order_date", "EEEE"))\
            .withColumn("day_part", 
            F.when((F.col("hour") >= 7) & (F.col("hour") < 12), "Breakfast")
            .when((F.col("hour") >= 12) & (F.col("hour") < 17), "Lunch")
            .when((F.col("hour") >= 17) & (F.col("hour") < 19), "Afternoon")
            .when((F.col("hour") >= 19) & (F.col("hour") < 22), "Dinner")
            .otherwise("Late Night")
            )

        # Order: Date key first, then hierarchy (Year -> Month -> Day)
        dim_date_df = dim_date_df.select(
            "order_date", "year", "month", "month_name", "day", "day_of_week_number", "day_name", "day_part", "is_weekend"
        )

        # write dim_date to gold layer
        dim_date_df.write.format("delta").mode("overwrite").saveAsTable(target_dim_date)

        print(f'>> Rows Loaded: {dim_date_df.count()}')
        print(f'>> Load Duration: {round(time.time() - start_time, 2)} seconds')

        # --- 4. Create Dimension: gold.dim_payment ---
        print('------------------------------------------------')
        print('Loading dim_payment Table')
        print('------------------------------------------------')
        start_time = time.time()

        df_payment = df_silver.select("payment_method").distinct()

        # Sequential Key and Column Ordering
        payment_window = Window.orderBy("payment_method")
        df_payment = df_payment.withColumn(
            "payment_skey", F.row_number().over(payment_window)
        )

        # Reorder: Key first, then payment method
        df_payment = df_payment.select("payment_skey", "payment_method")

        # write dim_payment to gold layer
        df_payment.write.format("delta").mode("overwrite").saveAsTable(target_dim_payment)

        print(f'>> Rows Loaded: {df_payment.count()}')
        print(f'>> Load Duration: {round(time.time() - start_time, 2)} seconds')

        # --- 5. Create Dimension: gold.dim_order_type ---
        print('------------------------------------------------')
        print('Loading dim_order_type Table')
        print('------------------------------------------------')
        start_time = time.time()

        df_order_type = df_silver.select("order_type").distinct()

        # Sequential Key and Column Ordering
        order_type_window = Window.orderBy("order_type")
        df_order_type = df_order_type.withColumn(
            "order_skey", F.row_number().over(order_type_window)
        )

        # Reorder: Key first, then order type
        df_order_type = df_order_type.select("order_skey", "order_type")

        # write dim_order_type to gold layer
        df_order_type.write.format("delta").mode("overwrite").saveAsTable(target_dim_order_type)

        print(f'>> Rows Loaded: {df_order_type.count()}')
        print(f'>> Load Duration: {round(time.time() - start_time, 2)} seconds')


       # --- 5.5 Create Dimension: gold.dim_customers ---
        print('------------------------------------------------')
        print('Loading dim_customers Table')
        print('------------------------------------------------')
        start_time_cust = time.time()

        # 1. تجميع البيانات
        customer_summary = df_silver.groupBy("customer_id").agg(
            F.countDistinct("order_id").alias("total_orders"),
            F.max("order_date").alias("last_order_date")
        )

        # 2. تاريخ المرجع
        ref_date = df_silver.select(F.max("order_date")).collect()[0][0]

        # 3. حساب الفجوة والتصنيف
        dim_customers_final = customer_summary.withColumn("days_gap", F.datediff(F.lit(ref_date), F.col("last_order_date"))) \
            .withColumn("customer_segment", 
            F.when(F.col("days_gap") > 90, "Lost")
            .when((F.col("days_gap") > 60) & (F.col("total_orders") >= 5), "At Risk")
            .when((F.col("total_orders") >= 60) & (F.col("days_gap") <= 7), "VIP")
            .when((F.col("total_orders") >= 15) & (F.col("days_gap") <= 30), "Loyal")
            .when(F.col("total_orders") < 15, "New")
            .otherwise("Slipping")
            )

        # 4. اختيار الأعمدة النهائية (الـ ID مع التصنيف)
        dim_customers_final = dim_customers_final.select(
            "customer_id", 
            "customer_segment", 
            "total_orders", 
            "last_order_date", 
            "days_gap"
        )

        # حفظ الجدول
        target_dim_customers = "restaurant_project.gold.dim_customers"
        dim_customers_final.write.format("delta").mode("overwrite").saveAsTable(target_dim_customers)

        print(f'>> Rows Loaded: {dim_customers_final.count()}')
        print(f'>> Load Duration: {round(time.time() - start_time_cust, 2)} seconds')



        # --- 6. Create Fact: gold.fact_orders ---
        print('------------------------------------------------')
        print('Loading fact_orders Table')
        print('------------------------------------------------')
        start_time = time.time()

        # Join to bring in Surrogate Keys
        fact_orders_df = df_silver \
            .join(dim_products_df, "item_name", "left")\
            .join(dim_branches_df, "branch", "left")\
            .join(df_payment, "payment_method", "left")\
            .join(df_order_type, "order_type", "left")
    
        # Column Ordering: Keys -> Date/Time -> Metrics (Quantity/Price)
        fact_orders_df = fact_orders_df.select(
            "sales_skey",       # Primary Fact Key
            "product_skey",     # Foreign Key to Product
            "branch_skey",      # Foreign Key to Branch
            "payment_skey",     # Foreign Key to Payment
            "order_skey",       # Foreign Key to Order Type
            "order_id",
            "customer_id",
            "order_date",       # Foreign Key to Date
            "hour",             # Time dimension
            "rating",           # metric 1 
            "quantity",         # Metric 2
            "price",            # Metric 3
            "discount",         # Metric 4
            "total_amount"      # Final Metric
        )
    
        # write fact_orders to gold layer
        fact_orders_df.write.format("delta").mode("overwrite").saveAsTable(target_fact_orders)

        print(f'>> Rows Loaded: {fact_orders_df.count()}')
        print(f'>> Load Duration: {round(time.time() - start_time, 2)} seconds')

        print('==========================================')
        print(f'Gold Layer Load Completed in {round(time.time() - batch_start_time, 2)} sec')
        print('==========================================')
    
    except Exception as e:
        print('==========================================')
        print('ERROR OCCURED DURING LOADING SILVER LAYER')
        print(f'Error Message: {str(e)}')
        print('==========================================')

# Execute the pipeline
load_gold_layer()